# ⚡ Altron Android APK Fast Builder (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/global-linguists-ai/Altron/blob/main/Altron_Android_Build_Colab.ipynb)

This official Google Colab notebook builds the standalone **Altron Android APK** directly using Google Cloud VM infrastructure.

> **Quick Start:** You can simply run **Section 1 (All-in-One Fast Build)** below. It will automatically clone the repository, install dependencies, configure Java 17 and the Android SDK, compile the release APK, and prompt your browser to download `app-release.apk`.

---

## 🚀 Option A: 1-Click Fast All-in-One Build (Recommended)
Press **Play** on this single cell to build and download the APK automatically.

In [ ]:
# =============================================================================
# 🤖 1-CLICK COMPLETE ALTRON ANDROID APK BUILD SCRIPT
# =============================================================================
import os, sys, glob, shutil

print("📥 Step 1/5: Cloning Altron repository from GitHub...")
!rm -rf /content/altron
!git clone --depth 1 https://github.com/global-linguists-ai/Altron.git /content/altron

print("📦 Step 2/5: Installing Node.js & React Native dependencies...")
%cd /content/altron
!npm install --legacy-peer-deps --no-audit --no-fund

print("☕ Step 3/5: Setting up OpenJDK 17 and Android SDK...")
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk wget unzip
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["ANDROID_HOME"] = "/content/android-sdk"
os.environ["PATH"] = f"{os.environ[JAVA_HOME]}/bin:/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:" + os.environ["PATH"]

# Download cmdline-tools if not present
if not os.path.exists("/content/android-sdk/cmdline-tools/latest"):
    !mkdir -p /content/android-sdk/cmdline-tools
    !wget -q -nc https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /content/cmdline-tools.zip
    !unzip -q -o /content/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
    !rm -rf /content/android-sdk/cmdline-tools/latest
    !mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest
    !yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null
    !/content/android-sdk/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-34" "build-tools;34.0.0" > /dev/null

# Configure local.properties
!mkdir -p /content/altron/android
!echo "sdk.dir=/content/android-sdk" > /content/altron/android/local.properties

print("🔨 Step 4/5: Compiling Standalone Android Release APK with Gradle...")
%cd /content/altron/android
!chmod +x ./gradlew
!./gradlew assembleRelease --no-daemon -x lint

print("💾 Step 5/5: Checking build output & downloading APK...")
from google.colab import files
apks = glob.glob("/content/altron/android/app/build/outputs/apk/**/*.apk", recursive=True)
if apks:
    print("\n=======================================================")
    print("🎉 BUILD COMPLETED SUCCESSFULLY!")
    print("=======================================================")
    for apk in apks:
        mb = os.path.getsize(apk) / (1024 * 1024)
        print(f" -> Found APK: {apk} ({mb:.2f} MB)")
    target_apk = [a for a in apks if "release" in a] or apks
    print(f"\n📥 Initiating browser download for: {target_apk[0]}")
    files.download(target_apk[0])
else:
    print("❌ Build completed but no APK was found. Check Gradle logs above.")


---
## 🛠️ Option B: Step-by-Step Execution (For Debugging or Customization)

### Step 1: Clone Repository
Clones the codebase from GitHub into Colab.

In [ ]:
!rm -rf /content/altron
!git clone --depth 1 https://github.com/global-linguists-ai/Altron.git /content/altron
%cd /content/altron
!ls -la

### Step 2: Install Node Dependencies
Installs packages with npm.

In [ ]:
%cd /content/altron
!npm install --legacy-peer-deps --no-audit --no-fund

### Step 3: Configure OpenJDK 17 and Android SDK Tools

In [ ]:
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk wget unzip
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["ANDROID_HOME"] = "/content/android-sdk"
os.environ["PATH"] = f"{os.environ["JAVA_HOME"]}/bin:/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:" + os.environ["PATH"]

!mkdir -p /content/android-sdk/cmdline-tools
!wget -q -nc https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /content/cmdline-tools.zip
!unzip -q -o /content/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
!rm -rf /content/android-sdk/cmdline-tools/latest
!mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest
!yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null
!/content/android-sdk/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-34" "build-tools;34.0.0" > /dev/null
!echo "sdk.dir=/content/android-sdk" > /content/altron/android/local.properties
!java -version

### Step 4: Run Gradle to Assemble Release APK

In [ ]:
%cd /content/altron/android
!chmod +x ./gradlew
!./gradlew assembleRelease --no-daemon -x lint

### Step 5: Download the Generated APK

In [ ]:
from google.colab import files
import glob, os

apks = glob.glob("/content/altron/android/app/build/outputs/apk/**/*.apk", recursive=True)
if apks:
    target_apk = [a for a in apks if "release" in a] or apks
    print(f"Downloading {target_apk[0]}...")
    files.download(target_apk[0])
else:
    print("❌ No APK found. Please inspect step 4.")